# AuditOwl on your own paper

<a href="https://colab.research.google.com/github/PascalIversen/auditowl-neurips2025/blob/main/examples/colab_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Fill in step 1, add `ANTHROPIC_API_KEY` or `CLAUDE_CODE_OAUTH_TOKEN` (from `claude setup-token`) under Secrets, then run all. The findings are based on AI and can be wrong! Check each one you act upon :)

`claude-opus-4-8` is the model we used (use `opus` for the latest Opus).

In [ ]:
# @title 1. Settings
PAPER_PDF_URL = ""  # @param {type:"string"}
REPO_URL = ""  # @param {type:"string"}
PAPER_NAME = "mypaper"  # @param {type:"string"}
AUDIT_MODEL = "claude-opus-4-8"  # @param ["claude-opus-4-8", "opus"] {allow-input: true}
VERIFY_MODEL = "claude-sonnet-4-6"  # @param ["claude-sonnet-4-6", "sonnet"] {allow-input: true}
MAX_BUDGET_USD = 40  # @param {type:"number"}

In [ ]:
# @title 2. Install AuditOwl and Claude Code
import os
%cd /content
!test -d auditowl-neurips2025 || git clone -q https://github.com/PascalIversen/auditowl-neurips2025.git
%cd /content/auditowl-neurips2025
!pip install -q pymupdf pyyaml
!curl -fsSL https://claude.ai/install.sh | bash
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!claude --version

In [ ]:
# @title 3. Credentials
from google.colab import userdata

for name in ("ANTHROPIC_API_KEY", "CLAUDE_CODE_OAUTH_TOKEN"):
    try:
        os.environ[name] = userdata.get(name)
        print(f"Using the {name} secret.")
        break
    except Exception:
        pass
else:
    raise RuntimeError("Add ANTHROPIC_API_KEY or CLAUDE_CODE_OAUTH_TOKEN under Secrets.")

In [ ]:
# @title 4. Fetch the paper and the code
import re, subprocess, sys
from pathlib import Path
import requests

sys.path.insert(0, "auditowl")
from fetch_paper_inputs import read_pdf_text

m = re.fullmatch(r"https?://github\.com/([\w.-]+)/([\w.-]+?)(?:\.git)?/?", REPO_URL.strip())
if not m:
    raise ValueError(f"REPO_URL is not a GitHub repository: {REPO_URL!r}")
owner, repo = m.groups()
paper = Path("audits") / PAPER_NAME
code_dir = paper / "code" / f"{owner}__{repo}"
code_dir.parent.mkdir(parents=True, exist_ok=True)

pdf = paper / "paper.pdf"
if PAPER_PDF_URL.strip():
    url = PAPER_PDF_URL.strip().replace("arxiv.org/abs/", "arxiv.org/pdf/")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    pdf.write_bytes(r.content)
else:
    from google.colab import files
    uploaded = files.upload()
    pdf.write_bytes(next(iter(uploaded.values())))
if not pdf.read_bytes().startswith(b"%PDF"):
    raise ValueError(f"{pdf} is not a PDF. Check PAPER_PDF_URL.")

text = read_pdf_text(str(pdf))[2]
(paper / "paper_text.txt").write_text(text, encoding="utf-8")
(paper / "metadata.txt").write_text(f"paper: {PAPER_PDF_URL.strip() or 'uploaded PDF'}\ncategory: empirical\n")
(paper / "code_links.txt").write_text(f"https://github.com/{owner}/{repo}\n")
if not code_dir.exists():
    subprocess.run(["git", "clone", "-q", f"https://github.com/{owner}/{repo}", str(code_dir)], check=True)

n_files = sum(1 for p in code_dir.rglob("*") if p.is_file() and ".git" not in p.parts)
print(f"Paper: {pdf} ({len(text):,} characters of text)")
print(f"Code:  {code_dir} ({n_files} files)")

In [ ]:
# @title 5. Run the audit
import json

def run_agent(prompt, model, log_path):
    cmd = ["claude", "-p", prompt, "--model", model,
           "--allowedTools", "Bash,Read,Write,Edit,Glob,Grep",
           "--max-budget-usd", str(MAX_BUDGET_USD),
           "--output-format", "stream-json", "--verbose"]
    result = None
    with open(log_path, "w") as log, subprocess.Popen(
            cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            log.write(line)
            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                print(line.rstrip())
                continue
            if event.get("type") == "assistant":
                for block in event["message"]["content"]:
                    if block.get("type") == "tool_use":
                        args = block["input"]
                        shown = args.get("command") or args.get("file_path") or args.get("pattern") or ""
                        shown = " ".join(str(shown).replace(os.getcwd(), ".").split())
                        print(f"[{block['name']}] {shown[:120]}")
            elif event.get("type") == "result":
                result = event
    if result is None:
        raise RuntimeError(f"The agent stopped without a result. The full log is in {log_path}.")
    print(f"\n{result['subtype']}: {result['num_turns']} turns, "
          f"{result['duration_ms'] / 60000:.1f} min, ${result['total_cost_usd']:.2f}")
    print(result.get("result") or "")
    return result

run_agent(f"""Audit the paper in `{paper}/` by following `auditowl/audit_prompt.md` exactly.
The code is under `code/{code_dir.name}/`. Write `audit.md`, then run
`python auditowl/extract_findings.py {paper}/audit.md --out {paper}/findings.json`.
Stay read-only on `code/`.""", AUDIT_MODEL, paper / "agent_audit.jsonl")

!python auditowl/extract_findings.py {paper}/audit.md --out {paper}/findings.json

In [ ]:
# @title 6. Verify the findings (optional)
run_agent(f"""Verify the audit in `{paper}/` by following `auditowl/verifier_prompt.md` exactly.
The original audit is `audit.md` and `findings.json`; the cloned repo is under `code/{code_dir.name}/`
(a finding's `file` is relative to that repo root). Stay read-only on `code/`, `_audit_code/`,
`audit.md` and `findings.json`; write only `findings_verified.json` and `_verifier_code/`.""",
          VERIFY_MODEL, paper / "agent_verify.jsonl")

In [ ]:
# @title 7. Read the report
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

src = paper / "findings_verified.json"
if not src.exists():
    src = paper / "findings.json"
df = pd.DataFrame(json.loads(src.read_text())["findings"])
if "verdict" in df:
    print(f"Verifier verdicts: {df['verdict'].value_counts().to_dict()}. Rejected findings are left out below.")
    df = df[df["verdict"] != "reject"]

if df.empty:
    print("No findings.")
else:
    counts = (pd.crosstab(df["category"], df["severity"])
                .reindex(index=["missing", "bug", "difference", "methodology"],
                         columns=["high", "medium", "low"], fill_value=0))
    ax = counts.plot.barh(stacked=True, color=["#c0392b", "#e6a23c", "#9aa5b1"], figsize=(6, 2.5))
    ax.set_xlabel("findings"); ax.set_ylabel(""); ax.invert_yaxis()
    plt.show()
    display(df[["category", "severity", "confidence", "title", "file", "line_start"]])

display(Markdown((paper / "audit.md").read_text()))

In [ ]:
# @title 8. Download the results
from google.colab import files

zip_path = f"/content/auditowl_{PAPER_NAME}.zip"
!cd audits && zip -qr {zip_path} {PAPER_NAME} -x "{PAPER_NAME}/code/*"
files.download(zip_path)